# RAG Pipeline (Notebook)

This notebook shows a complete pipeline to run a local-free retrieval augmented generation demo. Each code cell corresponds to the sections in the blueprint.

## 1. Environment setup (optional)

Install dependencies if needed. Run these once in your environment.

In [10]:
# %pip install -r requirements.txt  # Uncomment and run if needed

### Jupyter kernel / venv note

Make sure your Jupyter kernel is using the Python environment (venv) where you installed GPU-enabled PyTorch if you want to use CUDA. In VS Code: *Kernel > Change Kernel* and select the venv. Then re-run the notebook cells.

## 2. Imports and Device Setup

In [1]:
import os
from typing import List, Dict, Any
import torch
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from rag.device import get_device, cuda_diagnostics

device = get_device()
print("Using device:", device)
print(cuda_diagnostics())

d:\GABRIEL\Projects\research-rag-implementation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
{'torch_version': '2.7.1+cu118', 'cuda_version': '11.8', 'cuda_available': True, 'device_name': 'NVIDIA GeForce RTX 4060', 'total_memory': 8585216000, 'major': 8, 'minor': 9}


## 3. KB Data, Chunking, Embeddings and FAISS Index

In [2]:
raw_docs = [
    {"id": "doc_001", "text": "Retrieval-Augmented Generation (RAG) combines a retriever with a generator to answer questions based on external documents."},
    {"id": "doc_002", "text": "The BGE-small-en model is a compact, general-purpose English embedding model useful for semantic search and retrieval tasks."},
    {"id": "doc_003", "text": "FAISS is a library for efficient similarity search and clustering of dense vectors, widely used for vector databases."},
]

# iter_simple_chunk_text is not available in rag.utils in this environment.
# Use simple_chunk_text instead; it returns an iterable of chunks which we
# can wrap in list(...) for small demos.
from rag.utils import simple_chunk_text
kb_docs = []
for d in raw_docs:
    # For large documents you might want a streaming generator, but here we
    # collect chunks into a list for simplicity.
    chunks = list(simple_chunk_text(d['text'], max_chars=400, overlap=50))
    for idx, ch in enumerate(chunks):
        kb_docs.append({'id': f"{d['id']}_chunk_{idx}", 'text': ch})

print("Loaded", len(raw_docs), "raw docs. Chunked to", len(kb_docs), "docs.")

Loaded 3 raw docs. Chunked to 6 docs.


## 4. Embedding model & FAISS index

In [3]:
# Below code is optional; it loads a real embedding model if available.
# If you have limited VRAM or are offline, this step is optional for the notebook.
try:
    from sentence_transformers import SentenceTransformer
    emb_model = SentenceTransformer("BAAI/bge-small-en", device=device)
    texts = [d['text'] for d in kb_docs]
    embeddings = emb_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    embeddings = embeddings.astype('float32')
    import faiss
    dim = embeddings.shape[1]
    index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = 200
    index.hnsw.efSearch = 64
    index.add(embeddings)
    print("FAISS index created, size:", index.ntotal)
except Exception as e:
    print("Skipping embedding/index creation due to:", e)

# Optional: if you want to load a local LLM and you have a GPU, follow the README setup.
# Example LLM loader: check `device` and use device_map="auto" if accelerate is installed.
# from transformers import AutoTokenizer, AutoModelForCausalLM
# MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
# if device == 'cuda':
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=torch.float16)
# else:
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

FAISS index created, size: 6


## 5. Retriever and RAG Answer Flow

In [4]:
from rag.retriever import Retriever
from rag.device import get_device, cuda_diagnostics
device = get_device()
print("Device:", device)
retriever = Retriever(device=device)
retriever.build_index(kb_docs)
results = retriever.retrieve("What is RAG?", k=3)
for r in results:
    print(r['rank'], r['score'], r['id'], '->', r['text'][:80])

Device: cuda
0 0.859682559967041 doc_001_chunk_0 -> Retrieval-Augmented Generation (RAG) combines a retriever with a generator to an
1 0.8120253086090088 doc_002_chunk_1 -> el useful for semantic search and retrieval tasks.
2 0.7771009206771851 doc_003_chunk_1 -> f dense vectors, widely used for vector databases.
0 0.859682559967041 doc_001_chunk_0 -> Retrieval-Augmented Generation (RAG) combines a retriever with a generator to an
1 0.8120253086090088 doc_002_chunk_1 -> el useful for semantic search and retrieval tasks.
2 0.7771009206771851 doc_003_chunk_1 -> f dense vectors, widely used for vector databases.
